In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 8
set_seed(SEED)

# 加载数据
all_sequence_outputsnew=np.load('<RECIPE_PROJECT_ROOT>/data/bulk/mouse_sequence_known.npy')
merged_df = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/bulk/mouse_reference.csv')
#merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]

ppi_matrix = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/networks/mouse_ppi_known.csv')
ppi_matrix = sp.coo_matrix(ppi_matrix)


# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countsc18nc'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding


# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# 数据集划分
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

# 训练与验证
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 400 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 
for epoch in range(1, num_epochs + 1):
    # Training phase
    neural_net.train()
    optimizer.zero_grad()
    out, z = neural_net(data)
    train_loss = criterion(out[train_idx_X], data.y[train_idx_X]).mean()
    train_loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[train_idx_X].cpu().numpy(),
        out[train_idx_X].detach().cpu().numpy()
    )

    # Validation phase
    neural_net.eval()
    with torch.no_grad():
        val_out, _ = neural_net(data)
        val_loss = criterion(val_out[val_idx_X], data.y[val_idx_X]).mean()
        val_r2 = r2_score(data.y[val_idx_X].cpu().numpy(), val_out[val_idx_X].cpu().numpy())

        test_out, _ = neural_net(data)
        test_loss = criterion(test_out[test_idx_X], data.y[test_idx_X]).mean()
        test_r2 = r2_score(data.y[test_idx_X].cpu().numpy(), test_out[test_idx_X].cpu().numpy())

    # Check if validation loss has improved
    if val_r2 > best_val_r2:  # 使用 R² 作为早停指标
        best_val_r2 = val_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        patience_counter = 0

        best_test_loss = test_loss
        best_test_r2 = test_r2
        torch.save(neural_net.state_dict(), './models/bulk_mouse_unknown_seed8_best.pt')
        y_true_np = data.y[test_idx_X].cpu().detach().numpy()
        y_pred_np = test_out[test_idx_X].cpu().detach().numpy()


    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {best_val_loss:.3f}, Val R²: {best_val_r2:.3f}|"
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")

In [ ]:
# #保存模型
# #torch.save(neural_net, './models/bulk_mouse_unknown_seed8_full.pt')
# #保存模型
# torch.save(neural_net, './models/bulk_mouse_unknown_seed8_epoch679.pt')


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

# 加载保存的模型参数
state_dict = torch.load('./models/bulk_mouse_unknown_seed8_best.pt')
neural_net = NeuralGraph()  # 确保先定义模型实例
neural_net.load_state_dict(state_dict)
neural_net.eval()

#将模型和数据移至设备上
neural_net = neural_net.to(device)
data = data.to(device)

# 重新推理以获取输出
with torch.no_grad():
    out, _ = neural_net(data)

# 计算 R²、皮尔逊相关系数、斯皮尔曼相关系数和余弦相似度
y_true_np = data.y[test_idx_X].cpu().detach().numpy()
y_pred_np = out[test_idx_X].cpu().detach().numpy()

test_r2_kd = r2_score(y_true_np, y_pred_np)
pearson_r_kd, _ = pearsonr(y_true_np.flatten(), y_pred_np.flatten())
spearman_r_kd, _ = spearmanr(y_true_np.flatten(), y_pred_np.flatten())
cosine_sim_kd = cosine_similarity(y_true_np.reshape(1, -1), y_pred_np.reshape(1, -1))[0, 0]

print(f"KD Dataset - Pearson r: {pearson_r_kd:.3f}, Spearman r: {spearman_r_kd:.3f}, Cosine Similarity: {cosine_sim_kd:.3f}")
print(f"KD Dataset R²: {test_r2_kd:.3f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MaxAbsScaler

# 计算相关系数
correlation = np.corrcoef(y_true_np.flatten(), y_pred_np.flatten())[0, 1]


# 定义颜色
scatter_color = '#8d91c0'  # 散点图颜色
hist_color = '#b8acb9'     # 直方图颜色

# 创建图形和子图布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

# 散点图和直方图的布局
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 中间的散点图
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # 顶部的直方图
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # 右侧的直方图

# 绘制散点图
ax_scatter.scatter(y_true_np, y_pred_np, alpha=0.6, color=scatter_color)

# 绘制对角线作为参考
max_val = max(y_true_np.max(), y_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 添加回归线，不绘制散点
sns.regplot(x=y_true_np, y=y_pred_np, scatter=False, color='black', ax=ax_scatter)

# 设置标签和标题
ax_scatter.set_xlabel('True NC protein', fontsize=12)
ax_scatter.set_ylabel('Predict NC protein', fontsize=12)
ax_scatter.text(0.05, 0.9, f'correlation: {correlation:.4f}', transform=ax_scatter.transAxes)

# 绘制顶部和右侧的直方图
ax_histx.hist(y_true_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

# 去掉右侧直方图的 y 轴
ax_histy.yaxis.set_visible(False)

# 仅保留顶部直方图的 y 轴，去掉 x 轴
ax_histx.set_ylabel("Frequency")  # 仅显示 y 轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏 x 轴标签

# 显示最终图形
plt.show()


In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 12
set_seed(SEED)

# 加载数据
all_sequence_outputsnew=np.load('<RECIPE_PROJECT_ROOT>/data/bulk/mouse_sequence_known.npy')
merged_df = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/bulk/mouse_reference.csv')
#merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]

ppi_matrix = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/networks/mouse_ppi_known.csv')
ppi_matrix = sp.coo_matrix(ppi_matrix)


# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countsc18nc'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding


# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# 数据集划分
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

# 训练与验证
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 400 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 
for epoch in range(1, num_epochs + 1):
    # Training phase
    neural_net.train()
    optimizer.zero_grad()
    out, z = neural_net(data)
    train_loss = criterion(out[train_idx_X], data.y[train_idx_X]).mean()
    train_loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[train_idx_X].cpu().numpy(),
        out[train_idx_X].detach().cpu().numpy()
    )

    # Validation phase
    neural_net.eval()
    with torch.no_grad():
        val_out, _ = neural_net(data)
        val_loss = criterion(val_out[val_idx_X], data.y[val_idx_X]).mean()
        val_r2 = r2_score(data.y[val_idx_X].cpu().numpy(), val_out[val_idx_X].cpu().numpy())

        test_out, _ = neural_net(data)
        test_loss = criterion(test_out[test_idx_X], data.y[test_idx_X]).mean()
        test_r2 = r2_score(data.y[test_idx_X].cpu().numpy(), test_out[test_idx_X].cpu().numpy())

    # Check if validation loss has improved
    if val_r2 > best_val_r2:  # 使用 R² 作为早停指标
        best_val_r2 = val_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        patience_counter = 0

        best_test_loss = test_loss
        best_test_r2 = test_r2
        torch.save(neural_net.state_dict(), './models/bulk_mouse_unknown_seed12_best.pt')
        y_true_np = data.y[test_idx_X].cpu().detach().numpy()
        y_pred_np = test_out[test_idx_X].cpu().detach().numpy()


    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {best_val_loss:.3f}, Val R²: {best_val_r2:.3f}|"
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MaxAbsScaler

# 计算相关系数
correlation = np.corrcoef(y_true_np.flatten(), y_pred_np.flatten())[0, 1]


# 定义颜色
scatter_color = '#8d91c0'  # 散点图颜色
hist_color = '#b8acb9'     # 直方图颜色

# 创建图形和子图布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

# 散点图和直方图的布局
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 中间的散点图
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # 顶部的直方图
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # 右侧的直方图

# 绘制散点图
ax_scatter.scatter(y_true_np, y_pred_np, alpha=0.6, color=scatter_color)

# 绘制对角线作为参考
max_val = max(y_true_np.max(), y_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 添加回归线，不绘制散点
sns.regplot(x=y_true_np, y=y_pred_np, scatter=False, color='black', ax=ax_scatter)

# 设置标签和标题
ax_scatter.set_xlabel('True HCT116 protein', fontsize=12)
ax_scatter.set_ylabel('Predict HCT116 protein', fontsize=12)
ax_scatter.text(0.05, 0.9, f'correlation: {correlation:.4f}', transform=ax_scatter.transAxes)

# 绘制顶部和右侧的直方图
ax_histx.hist(y_true_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

# 去掉右侧直方图的 y 轴
ax_histy.yaxis.set_visible(False)

# 仅保留顶部直方图的 y 轴，去掉 x 轴
ax_histx.set_ylabel("Frequency")  # 仅显示 y 轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏 x 轴标签

# 显示最终图形
plt.show()


In [ ]:

test_r2_kd = r2_score(y_true_np, y_pred_np)
pearson_r_kd, _ = pearsonr(y_true_np.flatten(), y_pred_np.flatten())
spearman_r_kd, _ = spearmanr(y_true_np.flatten(), y_pred_np.flatten())
cosine_sim_kd = cosine_similarity(y_true_np.reshape(1, -1), y_pred_np.reshape(1, -1))[0, 0]

print(f"KD Dataset - Pearson r: {pearson_r_kd:.3f}, Spearman r: {spearman_r_kd:.3f}, Cosine Similarity: {cosine_sim_kd:.3f}")
print(f"KD Dataset R²: {test_r2_kd:.3f}")

In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 0
set_seed(SEED)

# 加载数据
all_sequence_outputsnew=np.load('<RECIPE_PROJECT_ROOT>/data/bulk/mouse_sequence_known.npy')
merged_df = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/bulk/mouse_reference.csv')
#merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]

ppi_matrix = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/networks/mouse_ppi_known.csv')
ppi_matrix = sp.coo_matrix(ppi_matrix)


# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countsc18nc'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding


# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# 数据集划分
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

# 训练与验证
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 400 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 
for epoch in range(1, num_epochs + 1):
    # Training phase
    neural_net.train()
    optimizer.zero_grad()
    out, z = neural_net(data)
    train_loss = criterion(out[train_idx_X], data.y[train_idx_X]).mean()
    train_loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[train_idx_X].cpu().numpy(),
        out[train_idx_X].detach().cpu().numpy()
    )

    # Validation phase
    neural_net.eval()
    with torch.no_grad():
        val_out, _ = neural_net(data)
        val_loss = criterion(val_out[val_idx_X], data.y[val_idx_X]).mean()
        val_r2 = r2_score(data.y[val_idx_X].cpu().numpy(), val_out[val_idx_X].cpu().numpy())

        test_out, _ = neural_net(data)
        test_loss = criterion(test_out[test_idx_X], data.y[test_idx_X]).mean()
        test_r2 = r2_score(data.y[test_idx_X].cpu().numpy(), test_out[test_idx_X].cpu().numpy())

    # Check if validation loss has improved
    if val_r2 > best_val_r2:  # 使用 R² 作为早停指标
        best_val_r2 = val_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        patience_counter = 0

        best_test_loss = test_loss
        best_test_r2 = test_r2
        torch.save(neural_net.state_dict(), './models/bulk_mouse_unknown_seed0_best.pt')
        y_true_np = data.y[test_idx_X].cpu().detach().numpy()
        y_pred_np = test_out[test_idx_X].cpu().detach().numpy()


    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {best_val_loss:.3f}, Val R²: {best_val_r2:.3f}|"
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")

In [ ]:

test_r2_kd = r2_score(y_true_np, y_pred_np)
pearson_r_kd, _ = pearsonr(y_true_np.flatten(), y_pred_np.flatten())
spearman_r_kd, _ = spearmanr(y_true_np.flatten(), y_pred_np.flatten())
cosine_sim_kd = cosine_similarity(y_true_np.reshape(1, -1), y_pred_np.reshape(1, -1))[0, 0]

print(f"KD Dataset - Pearson r: {pearson_r_kd:.3f}, Spearman r: {spearman_r_kd:.3f}, Cosine Similarity: {cosine_sim_kd:.3f}")
print(f"KD Dataset R²: {test_r2_kd:.3f}")

In [ ]:
# #保存模型
# #torch.save(neural_net, './models/bulk_mouse_unknown_seed8_full.pt')
# #保存模型
# torch.save(neural_net, './models/bulk_mouse_unknown_epoch1479.pt')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MaxAbsScaler

# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rKD2']].values / np.median(merged_df[['rKD2']].values))  + 1)
y_cpm_log2 = np.log2((merged_df[['KD3']].values / np.median(merged_df[['KD3']].values))  + 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countskd'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
kddata = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
kddata.pause = paired_ratio
kddata.seq = sequence_embedding

# 设置模型为评估模式
neural_net.eval()
neural_net.cpu()
kddata=kddata.cpu()
# 假设 y_true 和 val_out 已经被展平为一维张量
test_out, _ = neural_net(kddata)

# 将预测值和真实值展平
y_true = kddata.y[test_idx_X].view(-1)  # 将真实值转换为一维
y_pred = test_out[test_idx_X].view(-1)  # 将预测值转换为一维

# 转换为 NumPy 数组
y_true_np = y_true.cpu().detach().numpy()
y_pred_np = y_pred.cpu().detach().numpy()

# 计算相关系数
correlation = np.corrcoef(y_true_np, y_pred_np)[0, 1]


# 定义颜色
scatter_color = '#8d91c0'  # 散点图颜色
hist_color = '#b8acb9'     # 直方图颜色

# 创建图形和子图布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

# 散点图和直方图的布局
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 中间的散点图
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # 顶部的直方图
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # 右侧的直方图

# 绘制散点图
ax_scatter.scatter(y_true_np, y_pred_np, alpha=0.6, color=scatter_color)

# 绘制对角线作为参考
max_val = max(y_true_np.max(), y_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 添加回归线，不绘制散点
sns.regplot(x=y_true_np, y=y_pred_np, scatter=False, color='black', ax=ax_scatter)

# 设置标签和标题
ax_scatter.set_xlabel('True HCT116 protein', fontsize=12)
ax_scatter.set_ylabel('Predict HCT116 protein', fontsize=12)
ax_scatter.text(0.05, 0.9, f'correlation: {correlation:.4f}', transform=ax_scatter.transAxes)

# 绘制顶部和右侧的直方图
ax_histx.hist(y_true_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

# 去掉右侧直方图的 y 轴
ax_histy.yaxis.set_visible(False)

# 仅保留顶部直方图的 y 轴，去掉 x 轴
ax_histx.set_ylabel("Frequency")  # 仅显示 y 轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏 x 轴标签

# 显示最终图形
plt.show()


### 画图


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 12
set_seed(SEED)

# 加载数据
all_sequence_outputsnew = np.load('./data/all_sequence_outputs7132.npy')

merged_df = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]

ppi_matrix = pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_p.csv')
ppi_matrix = sp.coo_matrix(ppi_matrix)

pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_scribo_pause.csv')

pausing.columns = ['protein_id', "High_Pause_Countssc", "transcript_id"]
merged_df2 = pd.merge(merged_df, pausing, on='transcript_id', how='left')
merged_df2['High_Pause_Countssc'].fillna(0, inplace=True)

# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countsnc'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding


# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# 数据集划分
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

# 训练与验证
device = torch.device("cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 200 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

# 加载保存的模型参数
state_dict = torch.load('./models/bulk_mouse_unknown_best.pt')
neural_net = NeuralGraph()  # 确保先定义模型实例
neural_net.load_state_dict(state_dict)
neural_net.eval()

#将模型和数据移至设备上
neural_net = neural_net.to(device)
data = data.to(device)

# 重新推理以获取输出
with torch.no_grad():
    out, _ = neural_net(data)

# 计算 R²、皮尔逊相关系数、斯皮尔曼相关系数和余弦相似度
y_true_np = data.y[test_idx_X].cpu().detach().numpy()
y_pred_np = out[test_idx_X].cpu().detach().numpy()

test_r2_kd = r2_score(y_true_np, y_pred_np)
pearson_r_kd, _ = pearsonr(y_true_np.flatten(), y_pred_np.flatten())
spearman_r_kd, _ = spearmanr(y_true_np.flatten(), y_pred_np.flatten())
cosine_sim_kd = cosine_similarity(y_true_np.reshape(1, -1), y_pred_np.reshape(1, -1))[0, 0]

print(f"KD Dataset - Pearson r: {pearson_r_kd:.3f}, Spearman r: {spearman_r_kd:.3f}, Cosine Similarity: {cosine_sim_kd:.3f}")
print(f"KD Dataset R²: {test_r2_kd:.3f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MaxAbsScaler

# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rKD2']].values / np.median(merged_df[['rKD2']].values))  + 1)
y_cpm_log2 = np.log2((merged_df[['KD3']].values / np.median(merged_df[['KD3']].values))  + 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countskd'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
kddata = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
kddata.pause = paired_ratio
kddata.seq = sequence_embedding

# 设置模型为评估模式
neural_net.eval()
neural_net.cpu()
kddata=kddata.cpu()
# 假设 y_true 和 val_out 已经被展平为一维张量
test_out, _ = neural_net(kddata)

# 将预测值和真实值展平
y_true = kddata.y[test_idx_X].view(-1)  # 将真实值转换为一维
y_pred = test_out[test_idx_X].view(-1)  # 将预测值转换为一维

# 转换为 NumPy 数组
y_true_np = y_true.cpu().detach().numpy()
y_pred_np = y_pred.cpu().detach().numpy()

# 计算相关系数
correlation = np.corrcoef(y_true_np, y_pred_np)[0, 1]


# 定义颜色
scatter_color = '#8d91c0'  # 散点图颜色
hist_color = '#b8acb9'     # 直方图颜色

# 创建图形和子图布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

# 散点图和直方图的布局
ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # 中间的散点图
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)  # 顶部的直方图
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)  # 右侧的直方图

# 绘制散点图
ax_scatter.scatter(y_true_np, y_pred_np, alpha=0.6, color=scatter_color)

# 绘制对角线作为参考
max_val = max(y_true_np.max(), y_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 添加回归线，不绘制散点
sns.regplot(x=y_true_np, y=y_pred_np, scatter=False, color='black', ax=ax_scatter)

# 设置标签和标题
ax_scatter.set_xlabel('True HCT116 protein', fontsize=12)
ax_scatter.set_ylabel('Predict HCT116 protein', fontsize=12)
ax_scatter.text(0.05, 0.9, f'correlation: {correlation:.4f}', transform=ax_scatter.transAxes)

# 绘制顶部和右侧的直方图
ax_histx.hist(y_true_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

# 去掉右侧直方图的 y 轴
ax_histy.yaxis.set_visible(False)

# 仅保留顶部直方图的 y 轴，去掉 x 轴
ax_histx.set_ylabel("Frequency")  # 仅显示 y 轴标签
ax_histx.xaxis.set_visible(False)  # 隐藏 x 轴标签

# 显示最终图形
plt.show()


In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子
SEED = 12
set_seed(SEED)

# 加载数据
all_sequence_outputsnew = np.load('./data/all_sequence_outputs7132.npy')

merged_df = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df['protein'] = merged_df['protein_x'].str.split('.').str[0]

ppi_matrix = pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_p.csv')
ppi_matrix = sp.coo_matrix(ppi_matrix)

pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_scribo_pause.csv')

pausing.columns = ['protein_id', "High_Pause_Countssc", "transcript_id"]
merged_df2 = pd.merge(merged_df, pausing, on='transcript_id', how='left')
merged_df2['High_Pause_Countssc'].fillna(0, inplace=True)

# 数据标准化
# 数据标准化
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_standardized = scaler_X.fit_transform(merged_df[['rNC2']].values)
y_standardized = scaler_y.fit_transform(merged_df[['NC3']].values)

# 
# y = torch.tensor(np.array(merged_df['NC3'], dtype=np.float32).reshape(-1, 1))
# X = torch.from_numpy(np.array(merged_df['rNC2'], dtype=np.float32).reshape(-1, 1))

paired_ratio = torch.tensor(np.array(merged_df['High_Pause_Countsnc'], dtype=np.float32).reshape(-1, 1))

sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建图数据对象
data = Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)
data.pause = paired_ratio
data.seq = sequence_embedding


# 数据集划分
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# 设置随机种子8：1:1
#SEED = 12#Train R²: 0.866| Val Loss: 0.438, Val R²: 0.675|Test Loss: 0.576, Test R²: 0.379
#SEED = 24
set_seed(SEED)

# 数据集划分
X_train, X_temp, y_train, y_temp, train_idx_X, temp_idx_X = train_test_split(
    X, y, np.arange(len(X)), test_size=0.25, random_state=SEED)

X_val, X_test, y_val, y_test, val_idx_X, test_idx_X = train_test_split(
    X_temp, y_temp, temp_idx_X, test_size=1/3, random_state=SEED)

sequence_embedding_train, sequence_embedding_temp, train_idx_seq, temp_idx_seq = train_test_split(
    sequence_embedding, np.arange(len(sequence_embedding)), test_size=0.25, random_state=SEED)

sequence_embedding_val, sequence_embedding_test, val_idx_seq, test_idx_seq = train_test_split(
    sequence_embedding_temp, temp_idx_seq, test_size=1/3, random_state=SEED)

pause_train, pause_temp, train_idx_pause, temp_idx_pause = train_test_split(
    paired_ratio, np.arange(len(paired_ratio)), test_size=0.25, random_state=SEED)

pause_val, pause_test, val_idx_pause, test_idx_pause = train_test_split(
    pause_temp, temp_idx_pause, test_size=1/3, random_state=SEED)

assert np.array_equal(train_idx_X, train_idx_seq) and np.array_equal(train_idx_X, train_idx_pause), "训练集索引不一致"
assert np.array_equal(val_idx_X, val_idx_seq) and np.array_equal(val_idx_X, val_idx_pause), "验证集索引不一致"
assert np.array_equal(test_idx_X, test_idx_seq) and np.array_equal(test_idx_X, test_idx_pause), "测试集索引不一致"

class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)

    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        
        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore)), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        return out, z

# 训练与验证
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=7e-2)
criterion = nn.MSELoss()
data = data.to(device)

patience = 1000 #试试更大的early stopping #300-0.495
num_epochs = 3000
patience_counter = 0
best_val_loss = float('inf')
best_val_r2 = float('-inf') 
best_test_r2 = float('-inf') 
for epoch in range(1, num_epochs + 1):
    # Training phase
    neural_net.train()
    optimizer.zero_grad()
    out, z = neural_net(data)
    train_loss = criterion(out[train_idx_X], data.y[train_idx_X]).mean()
    train_loss.backward()
    optimizer.step()

    train_r2 = r2_score(
        data.y[train_idx_X].cpu().numpy(),
        out[train_idx_X].detach().cpu().numpy()
    )

    # Validation phase
    neural_net.eval()
    with torch.no_grad():
        val_out, _ = neural_net(data)
        val_loss = criterion(val_out[val_idx_X], data.y[val_idx_X]).mean()
        val_r2 = r2_score(data.y[val_idx_X].cpu().numpy(), val_out[val_idx_X].cpu().numpy())

        test_out, _ = neural_net(data)
        test_loss = criterion(test_out[test_idx_X], data.y[test_idx_X]).mean()
        test_r2 = r2_score(data.y[test_idx_X].cpu().numpy(), test_out[test_idx_X].cpu().numpy())

    # Check if validation loss has improved
    if val_r2 > best_val_r2:  # 使用 R² 作为早停指标
        best_val_r2 = val_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        patience_counter = 0

        best_test_loss = test_loss
        best_test_r2 = test_r2


    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {best_val_loss:.3f}, Val R²: {best_val_r2:.3f}|"
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")